# Web and social media analysis — data collection
A project of social media analysis about these questions:

1. **Sentiment & framing** — Do people talk about MT as a tool, a threat, or both? How does framing differ between users (translators vs. general public vs. tech advocates)?
2. **Professional impact discourse** — How do working translators and localizers discuss MT's effect on their careers, rates, and job availability?
3. **Utility vs. displacement tension** — Do people acknowledge MT's utility while also lamenting professional erosion, or are these largely separate conversations?
4. **Network structure** — Do professional translators and general users form distinct communities in the follow/reply networks, and which accounts bridge them?

## Setup

In [1]:
!pip -q install pandas atproto

In [2]:
import time
import re
from datetime import datetime, timezone, timedelta
import pandas as pd

from atproto import Client, models
from atproto_client.exceptions import RequestException, InvokeTimeoutError, NetworkError
import os

In [3]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/bluesky-data/'
os.makedirs(DATA_DIR, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Data gathering

Use BlueSky social media, their API to fetch posts, threads, users and crawl to get connections.

In [4]:
from google.colab import userdata

client = Client()
client.login(userdata.get("BSKY_HANDLE"), userdata.get("BSKY_PASSWORD"))
print("logged in")

logged in


In [5]:
# SearchPostsResponse
# │
# ├── posts: List[PostView]
# │   │
# │   └── PostView
# │       ├── uri          # unique ID of the post (e.g. at://did:plc:.../post/abc123)
# │       ├── cid          # content hash
# │       ├── like_count
# │       ├── repost_count
# │       ├── reply_count
# │       │
# │       ├── author: ProfileViewBasic
# │       │   ├── handle       # e.g. "janedoe.bsky.social"
# │       │   ├── display_name
# │       │   └── description  # ← this is the bio, useful for user labeling
# │       │
# │       └── record: Record   # ← the actual post content
# │           ├── text         # ← the post text you want
# │           ├── created_at   # timestamp
# │           └── reply: Optional
# │               ├── parent.uri   # uri of the post this replies to
# │               └── root.uri     # uri of the thread root
# │
# └── cursor   # pagination token — pass this to get the next page of results

Data gathering pipeline:

1. Search keywords → seed posts
2. Crawl threads of each seed post → more posts (reply structure comes from each post's `reply_to`)
3. Crawl **followers** of top authors → follow edges; fetch profiles of **all** authors → bios for user labeling
4. Deduplicate everything → final dataset

In [6]:
def _now_utc_ts() -> int:
    return int(datetime.now(timezone.utc).timestamp())


def call_with_retries(callable_fn, *args, max_retries: int = 10, **kwargs):
    for attempt in range(max_retries):
        try:
            return callable_fn(*args, **kwargs)
        except RequestException as e:
            resp = getattr(e, "response", None)
            status = getattr(resp, "status_code", None)
            headers = getattr(resp, "headers", {}) or {}

            if status == 429:
                reset = headers.get("ratelimit-reset") or headers.get("RateLimit-Reset")
                retry_after = headers.get("retry-after") or headers.get("Retry-After")

                if reset:
                    wait_s = max(1, int(reset) - _now_utc_ts()) + 1
                elif retry_after:
                    wait_s = int(float(retry_after))
                else:
                    wait_s = min(60, 2 ** attempt)

                print(f"[429] rate limited, sleeping {wait_s}s")
                time.sleep(wait_s)
                continue

            wait_s = min(10, 2 ** attempt)
            print(f"[{status}] request failed, sleeping {wait_s}s")
            time.sleep(wait_s)

    raise RuntimeError("Too many retries / repeated failures.")

In [7]:
# Facet extraction: hashtags, mentions, links

def extract_facets(record) -> dict:
    """Extract hashtags, mentions, links from a post record (if facets exist)."""
    hashtags = []
    mentions = []
    links = []

    facets = getattr(record, "facets", None) or []
    for facet in facets:
        features = getattr(facet, "features", None) or []
        for feat in features:
            ftype = getattr(feat, "py_type", None) or getattr(feat, "type", None)

            if ftype == "app.bsky.richtext.facet#tag":
                tag = getattr(feat, "tag", None)
                if tag:
                    hashtags.append(tag)
            elif ftype == "app.bsky.richtext.facet#mention":
                did = getattr(feat, "did", None)
                if did:
                    mentions.append({"did": did})
            elif ftype == "app.bsky.richtext.facet#link":
                uri = getattr(feat, "uri", None)
                if uri:
                    links.append(uri)

    return {"hashtags": hashtags, "mentions": mentions, "links": links}

In [8]:
# Search function with time window and facets extraction
def _parse_dt_utc(iso_str: str) -> datetime:
    dt = datetime.fromisoformat(iso_str.replace("Z", "+00:00"))
    if dt.tzinfo is None:
        return dt.replace(tzinfo=timezone.utc)
    return dt.astimezone(timezone.utc)

def search_posts_time_window(
    client,
    query: str = None,
    tag: str = None,
    since_iso: str = None,
    until_iso: str = None,
    max_posts: int = 1000,
    page_size: int = 50,
    polite_sleep: float = 0.25,
    print_every_page: bool = False,
):
    cursor = None
    rows = []
    page = 0

    while len(rows) < max_posts:
        page += 1

        params = models.AppBskyFeedSearchPosts.Params(
            q=query if query else tag,
            tag=[tag] if tag else None,
            sort="latest",
            since=since_iso,
            until=until_iso,
            limit=min(page_size, max_posts - len(rows)),
            cursor=cursor,
        )

        res = call_with_retries(client.app.bsky.feed.search_posts, params)
        cursor = res.cursor
        posts = res.posts or []

        if not posts:
            if print_every_page:
                print(f"[page {page}] no posts, stopping.")
            break

        if print_every_page:
            dts = []
            for p in posts:
                created = getattr(getattr(p, 'record', None), 'created_at', None)
                if created:
                    dts.append(_parse_dt_utc(created))
            if dts:
                print(
                    f"[page {page}] newest={max(dts).isoformat()}  oldest={min(dts).isoformat()}  "
                    f"collected={len(rows)}  cursor={'yes' if cursor else 'no'}"
                )

        for p in posts:
            rec = getattr(p, "record", None)
            created = getattr(rec, "created_at", None)
            if not created:
                continue

            created_dt = _parse_dt_utc(created).strftime("%Y-%m-%dT%H:%M:%S.000Z")
            if not (since_iso <= created_dt < until_iso):
                continue

            author = getattr(p, "author", None)
            facets_data = extract_facets(rec)
            reply = getattr(rec, "reply", None)

            rows.append({
                "created_at": created_dt,
                "uri": p.uri,
                "cid": getattr(p, "cid", None),
                "text": getattr(rec, "text", None),
                "handle": getattr(author, "handle", None),
                "did": getattr(author, "did", None),
                "replies": getattr(p, "reply_count", None),
                "likes": getattr(p, "like_count", None),
                "reposts": getattr(p, "repost_count", None),
                "quotes": getattr(p, "quote_count", None),
                "hashtags": facets_data["hashtags"],
                "mentions": facets_data["mentions"],
                "links": facets_data["links"],
                "is_reply": reply is not None,
                "reply_to": reply.parent.uri if reply else None,
                "query": query or f"#{tag}",
            })

            if len(rows) >= max_posts:
                break

        if cursor is None:
            if print_every_page:
                print(f"[page {page}] cursor is None, stopping.")
            break

        time.sleep(polite_sleep)

    df = pd.DataFrame(rows)
    if not df.empty:
        df["created_at"] = pd.to_datetime(df["created_at"], utc=True, errors="coerce")
        df = df.sort_values("created_at", ascending=True).reset_index(drop=True)
    return df

In [9]:
def get_followers_list(client, actor: str, max_total: int = 300, page_size: int = 100, polite_sleep: float = 0.6):
    cursor = None
    out = []
    while len(out) < max_total:
        params = models.AppBskyGraphGetFollowers.Params(
            actor=actor,
            limit=min(page_size, max_total - len(out)),
            cursor=cursor
        )
        res = call_with_retries(client.app.bsky.graph.get_followers, params)
        cursor = res.cursor
        followers = res.followers or []

        out.extend([{
            "handle": getattr(f, "handle", None),
            "did": getattr(f, "did", None),
            "display_name": getattr(f, "display_name", None),
            "description": getattr(f, "description", None),
        } for f in followers])

        if cursor is None or not followers:
            break
        time.sleep(polite_sleep)
    return out

In [10]:
# Time window: last 12 months
UNTIL_ISO = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S.000Z")
SINCE_ISO = (datetime.now(timezone.utc) - timedelta(days=365)).strftime("%Y-%m-%dT%H:%M:%S.000Z")

print(f"Collecting from {SINCE_ISO} to {UNTIL_ISO}")

### Seed Crawl

In [11]:
# Define queries and hastags to start from
TEXT_QUERIES = [
    '"machine translation"',
    '"AI translation"',
    '"post-editing"',
    '"human translation"',
    '"translation quality"',
    '"translator jobs"',
    '"translation industry"',
    'DeepL',
    'MTPE',
    'neural machine translation',
]

COMPOUND_QUERIES = [
    '"post-editing" AI',
    '"AI translation" threat',
    'scraped translation',
    '"machine translation" slop',
    '"human translator" AI',
    '"translation industry" AI',
]

HASHTAG_QUERIES = [
    'machinetranslation',
    'MTPE',
    'localization',
    'l10n',
    'SlowTranslation',
    'HumanTranslation',
    'PauseAI',
    'NoAI',
]

In [12]:
# API data fetching

all_dfs = []

for q in TEXT_QUERIES + COMPOUND_QUERIES:
    df_q = search_posts_time_window(
        client, query=q,
        since_iso=SINCE_ISO, until_iso=UNTIL_ISO,
        max_posts=300, page_size=25, polite_sleep=0.35,
    )
    df_q['query'] = q
    all_dfs.append(df_q)

for tag in HASHTAG_QUERIES:
    df_t = search_posts_time_window(
        client, tag=tag,
        since_iso=SINCE_ISO, until_iso=UNTIL_ISO,
        max_posts=300, page_size=25, polite_sleep=0.35,
    )
    df_t['query'] = f'#{tag}'
    all_dfs.append(df_t)

df_seeds = pd.concat(all_dfs, ignore_index=True).drop_duplicates(subset='uri').reset_index(drop=True)
print(f"\nTotal seed posts: {len(df_seeds)}")
print(f"Unique authors: {df_seeds['handle'].nunique()}")
print(f"\nBreakdown:\n{df_seeds['query'].value_counts()}")


Total seed posts: 3751
Unique authors: 2394

Breakdown:
query
"machine translation"         300
#NoAI                         300
#l10n                         300
#localization                 299
"AI translation"              297
MTPE                          295
DeepL                         294
"post-editing"                293
"human translation"           290
"translation quality"         275
#PauseAI                      263
"translation industry"        142
neural machine translation    108
"human translator" AI          82
"post-editing" AI              77
#machinetranslation            67
"machine translation" slop     22
scraped translation            21
"translator jobs"              17
#HumanTranslation               6
"AI translation" threat         3
Name: count, dtype: int64


In [13]:
df_seeds.to_csv(DATA_DIR + 'seeds.csv', index=False)
print("Saved seeds.csv")

Saved seeds.csv


### Thread Crawl

In [14]:
# Crawl the threads of seed posts.
# One get_post_thread call per seed (depth=10 returns the whole reply
# subtree), then walk the tree in memory — far fewer API calls than
# fetching every reply individually, and it goes through
# call_with_retries so rate limits don't silently drop threads.
#
# NOTE: reply EDGES are no longer collected here. Every post already
# stores its parent in `reply_to`, so edges are derived from the posts
# table in preprocessing — that also captures seed posts that are
# themselves replies, which a crawl-time list would miss.

visited_uris = set(df_seeds['uri'].tolist())  # skip re-fetching the seeds
thread_posts = []

def walk_thread(node):
    post = getattr(node, 'post', None)
    if post is None:
        return
    if post.uri not in visited_uris:
        visited_uris.add(post.uri)
        facets = extract_facets(post.record)
        thread_posts.append({
            'uri':        post.uri,
            'text':       post.record.text,
            'created_at': post.record.created_at,
            'handle':     post.author.handle,
            'did':        post.author.did,
            'likes':      post.like_count,
            'reposts':    post.repost_count,
            'replies':    post.reply_count,
            'is_reply':   post.record.reply is not None,
            'reply_to':   post.record.reply.parent.uri if post.record.reply else None,
            'hashtags':   facets['hashtags'],
            'mentions':   facets['mentions'],
            'links':      facets['links'],
            'query':      'thread_crawl',
        })
    for reply in (getattr(node, 'replies', None) or []):
        walk_thread(reply)

# only crawl seeds that have replies — no point crawling dead threads
seeds_with_replies = df_seeds[df_seeds['replies'] > 0]['uri'].tolist()
print(f"Seeds with replies: {len(seeds_with_replies)} / {len(df_seeds)}")

for i, uri in enumerate(seeds_with_replies):
    if i % 100 == 0:
        print(f"  crawling {i}/{len(seeds_with_replies)}...")
    try:
        thread = call_with_retries(client.app.bsky.feed.get_post_thread,
                                   {'uri': uri, 'depth': 10})
        walk_thread(thread.thread)
    except Exception as e:
        print(f"  skipped {uri}: {e}")
    time.sleep(0.3)

df_threads = pd.DataFrame(thread_posts).drop_duplicates(subset='uri').reset_index(drop=True)
print(f"\nNew posts from threads: {len(df_threads)}")

Seeds with replies: 1516 / 3751
  crawling 0/1516...
  crawling 100/1516...
  crawling 200/1516...
  crawling 300/1516...
  crawling 400/1516...
  crawling 500/1516...
  crawling 600/1516...
  crawling 700/1516...
  crawling 800/1516...
  crawling 900/1516...
  crawling 1000/1516...
  crawling 1100/1516...
  crawling 1200/1516...
  crawling 1300/1516...
  crawling 1400/1516...
  crawling 1500/1516...

New posts from threads: 5598


In [15]:
df_all_posts = pd.concat([df_seeds, df_threads], ignore_index=True)
df_all_posts = df_all_posts.drop_duplicates(subset='uri').reset_index(drop=True)

# Normalize created_at: seeds are already datetimes, thread posts are raw
# ISO strings — parse everything to UTC so sorting/temporal analysis works.
df_all_posts['created_at'] = pd.to_datetime(df_all_posts['created_at'], utc=True, errors='coerce', format='mixed')

print(f"Total posts: {len(df_all_posts)}")
print(f"Unique authors: {df_all_posts['handle'].nunique()}")
print(f"Date range: {df_all_posts['created_at'].min()} → {df_all_posts['created_at'].max()}")

df_all_posts.to_csv(DATA_DIR + 'posts_raw.csv', index=False)
print("Saved posts_raw.csv")

Total posts: 9349
Unique authors: 4220
Date range: 2025-07-06 20:38:38+00:00 → 2026-07-06 08:47:37+00:00
Saved posts_raw.csv


### Author / Follower Crawl

In [16]:
# Only crawl the followers of the most active users, not all
author_counts = df_all_posts['handle'].value_counts()
top_authors = author_counts[author_counts >= 2].index.tolist()
print(f"Authors with at least 2 posts: {len(top_authors)}")

Authors with at least 2 posts: 1573


In [17]:
# ── FOLLOWER CRAWL ──────────────────────────────────────────────
# For each top author, fetch up to FOLLOWERS_CAP of their FOLLOWERS.
# Edge direction: source follows target.
#
# LIMITATION: FOLLOWERS_CAP truncates the
# follower lists of popular accounts — it shows up as a spike at the cap
# in the in-degree distribution. Raising it costs one extra API page per
# +100 followers per author.
FOLLOWERS_CAP = 100

follow_edges = []

for i, handle in enumerate(top_authors):
    try:
        followers = get_followers_list(client, handle, max_total=FOLLOWERS_CAP,
                                       page_size=100, polite_sleep=0.3)
        for f in followers:
            follow_edges.append({
                'source': f['handle'],
                'target': handle,
                'type':   'follows',
            })
        time.sleep(0.3)
        if i % 100 == 0:
            print(f"  crawling {i}/{len(top_authors)}...")
    except Exception as e:
        print(f"  skipped {handle}: {e}")

df_follow_edges = pd.DataFrame(follow_edges)
print(f"\nFollow edges collected: {len(df_follow_edges)}")

df_follow_edges.to_csv(DATA_DIR + 'follow_edges.csv', index=False)
print("Saved follow_edges.csv")

  crawling 0/1573...
  crawling 100/1573...
  crawling 200/1573...
  crawling 300/1573...
  crawling 400/1573...
  crawling 500/1573...
  crawling 600/1573...
  crawling 700/1573...
  crawling 800/1573...
  crawling 900/1573...
  crawling 1000/1573...
  crawling 1100/1573...
  crawling 1200/1573...
  crawling 1300/1573...
  crawling 1400/1573...
  crawling 1500/1573...

Follow edges collected: 127890
Saved follow_edges.csv


In [18]:
# ── PROFILES FOR ALL POST AUTHORS ───────────────────────────────
# Bios are needed for user-type labeling. Fetching them only for top
# authors would leave everyone else without a bio, and they'd end up
# silently labeled 'general'. get_profiles accepts 25 actors per call,
# so covering all authors is cheap.
all_authors = [h for h in df_all_posts['handle'].dropna().unique()
               if not h.endswith('.invalid')]
print(f"Fetching profiles for {len(all_authors)} authors "
      f"({(len(all_authors) + 24) // 25} batched calls)")

author_profiles = []
for i in range(0, len(all_authors), 25):
    batch = all_authors[i:i + 25]
    try:
        res = call_with_retries(
            client.app.bsky.actor.get_profiles,
            models.AppBskyActorGetProfiles.Params(actors=batch)
        )
        for p in res.profiles:
            author_profiles.append({
                'handle':       p.handle,
                'did':          p.did,
                'display_name': p.display_name,
                'bio':          p.description,
                'followers':    p.followers_count,
                'following':    p.follows_count,
                'posts_count':  p.posts_count,
            })
    except Exception as e:
        print(f"  skipped batch {i // 25}: {e}")
    time.sleep(0.2)

df_profiles = pd.DataFrame(author_profiles).drop_duplicates(subset='handle')
print(f"\nProfiles collected: {len(df_profiles)}")

df_profiles.to_csv(DATA_DIR + 'author_profiles.csv', index=False)
print("Saved author_profiles.csv")

Fetching profiles for 4219 authors (169 batched calls)

Profiles collected: 4219
Saved author_profiles.csv


## Preprocessing
Deduplicating and cleaning. Runs entirely on the saved raw files (no API calls), and is **idempotent** — every output is rebuilt from the raw inputs, so re-running any cell cannot corrupt the data (the previous version resolved reply URIs in-place, which produced 0 edges when the cell ran twice).

The corpus is **multilingual** (search queries are English but threads contain other languages); we keep non-English posts and note it in the report, rather than claiming an English-only corpus.

In [ ]:
# RUN FROM HERE ON SAVED RAW DATA - not calling API anymore, now local cleaning
df = pd.read_csv(DATA_DIR + 'posts_raw.csv')
df_profiles = pd.read_csv(DATA_DIR + 'author_profiles.csv')
df_follow_edges = pd.read_csv(DATA_DIR + 'follow_edges.csv')

df['created_at'] = pd.to_datetime(df['created_at'], utc=True, errors='coerce', format='mixed')

print(f"Posts loaded: {len(df)}")
print(f"Columns: {list(df.columns)}")

In [20]:
before = len(df)
df = df.drop_duplicates(subset='uri').reset_index(drop=True)
print(f"Dropped {before - len(df)} duplicates, {len(df)} remaining")

Dropped 0 duplicates, 9349 remaining


In [21]:
# ── REPLY EDGES (derived from the posts table, BEFORE filtering) ──
# Every post stores its parent URI in reply_to, so reply edges come
# straight from the raw posts — including seeds that are replies.
# Targets are resolved against the RAW posts so that replies to posts
# later filtered as off-topic still resolve to a handle.
uri_to_handle = dict(zip(df['uri'], df['handle']))

replies = df[df['reply_to'].notna()]
df_reply_edges = pd.DataFrame({
    'source':     replies['handle'],
    'target_uri': replies['reply_to'],
})
df_reply_edges['target'] = df_reply_edges['target_uri'].map(uri_to_handle)
df_reply_edges['type'] = 'reply'

n_total = len(df_reply_edges)
n_resolved = df_reply_edges['target'].notna().sum()
print(f"Reply edges: {n_total} ({n_resolved} resolved to a handle, "
      f"{n_total - n_resolved} point outside the corpus)")

df_reply_edges.to_csv(DATA_DIR + 'reply_edges.csv', index=False)
print("Saved reply_edges.csv")

Reply edges: 7116 (5840 resolved to a handle, 1276 point outside the corpus)
Saved reply_edges.csv


In [22]:
# ── REMOVE PLACEHOLDER ACCOUNTS ────────────────────────────────
# 'handle.invalid' is Bluesky's placeholder for deleted/unresolvable
# accounts — not a real user.
def is_valid_handle(h):
    return isinstance(h, str) and not h.endswith('.invalid')

before = len(df)
df = df[df['handle'].map(is_valid_handle)].reset_index(drop=True)
df_follow_edges = df_follow_edges[
    df_follow_edges['source'].map(is_valid_handle)
    & df_follow_edges['target'].map(is_valid_handle)
].reset_index(drop=True)
df_reply_edges = df_reply_edges[df_reply_edges['source'].map(is_valid_handle)]
df_reply_edges = df_reply_edges[
    df_reply_edges['target'].isna() | df_reply_edges['target'].map(is_valid_handle)
].reset_index(drop=True)

print(f"Posts dropped (invalid handle): {before - len(df)}")

Posts dropped (invalid handle): 10


In [23]:
def clean_text(text):
    text = str(text)
    text = re.sub(r'http\S+', '', text)       # remove URLs
    text = re.sub(r'\s+', ' ', text).strip()  # normalize whitespace
    return text

df['text_clean'] = df['text'].apply(clean_text)
df = df[df['text_clean'].str.len() > 10].reset_index(drop=True)
print(f"Posts after empty drop: {len(df)}")

Posts after empty drop: 8992


In [24]:
# Topical relevance filtering
MT_TERMS = [
    'translat', 'interpret', 'locali', 'localiz',  # root forms catch variants
    'deepl', 'mtpe', 'post-edit', 'post edit',
    'language model', 'nlp', 'multilingual',
    'linguist', 'terminology', 'glossary',
]

def is_relevant(text):
    text_lower = text.lower()
    return any(term in text_lower for term in MT_TERMS)

before = len(df)
df = df[df['text_clean'].apply(is_relevant)].reset_index(drop=True)
print(f"Dropped {before - len(df)} irrelevant posts")
print(f"Remaining: {len(df)}")

Dropped 5119 irrelevant posts
Remaining: 3873


In [25]:
# ── KEYWORD-COLLISION FILTER ───────────────────────────────────
# 'MTPE' also matches Peru's Ministry of Labor (Ministerio de Trabajo y
# Promoción del Empleo), whose posts have nothing to do with machine
# translation. Conservative filter: requires MTPE AND Spanish
# employment/ministry terms in the same post.
peru_terms = r'(?:empleo|laboral|vacante|ministerio|piura|per[uú])'
noise_pattern = rf'(?i)(?:mtpe.*{peru_terms}|{peru_terms}.*mtpe)'
noise_mask = df['text_clean'].fillna('').str.contains(noise_pattern, regex=True)
print(f"Posts dropped (MTPE-ministry noise): {noise_mask.sum()}")
for t in df.loc[noise_mask, 'text_clean'].head(3):
    print(f"  x {t[:100]}")
df = df[~noise_mask].reset_index(drop=True)
print(f"Remaining: {len(df)}")

Posts dropped (MTPE-ministry noise): 9
  x Encuentro muchísimo más digno que digan "yo hago MTPE porque necesito plata" a que traten de darle v
  x Ofrecerán más de 240 oportunidades laborales en Ecco, Topitop y otras empresas reconocidas este vier
  x Piura: feria del empleo del MTPE ofrecerá más de 2.000 vacantes laborales elcomercio.pe/peru/piura/p
Remaining: 3864


In [26]:
# ── USER TYPE LABELING ─────────────────────────────────────────
# Based on profile bios. Authors whose profile/bio was not collected are
# labeled 'unknown' — NOT 'general' — so that downstream comparisons of
# user types aren't polluted by unlabeled accounts.

PROFESSIONAL_TERMS = [
    'translat', 'interpret', 'locali', 'linguist',
    'terminolog', 'lexicograph', 'subtitl', 'transcrib'
]

TECH_TERMS = [
    'developer', 'engineer', 'nlp', 'machine learning', 'ai researcher',
    'data scientist', 'software', 'programmer', 'coder', 'researcher'
]

def label_user(bio):
    if not isinstance(bio, str) or not bio.strip():
        return 'unknown'   # no bio collected — don't assume 'general'
    bio = bio.lower()
    if any(t in bio for t in PROFESSIONAL_TERMS):
        return 'professional'
    if any(t in bio for t in TECH_TERMS):
        return 'tech'
    return 'general'

bio_map = dict(zip(df_profiles['handle'], df_profiles['bio']))
has_profile = set(df_profiles['handle'])

df['bio'] = df['handle'].map(bio_map)
# distinguish "no profile fetched" from "profile fetched, empty bio"
df['user_type'] = [
    label_user(bio) if handle in has_profile else 'unknown'
    for handle, bio in zip(df['handle'], df['bio'])
]

print(df['user_type'].value_counts())
print(f"\nAuthors without a fetched profile: "
      f"{(~df['handle'].isin(has_profile)).sum()} posts")

user_type
general         2843
professional     610
unknown          218
tech             193
Name: count, dtype: int64

Authors without a fetched profile: 0 posts


In [27]:
# ── BUILD graph.csv (idempotent: rebuilt fresh from raw inputs) ─
df_reply_resolved = (df_reply_edges.dropna(subset=['target'])
                     [['source', 'target', 'type']])
df_graph = pd.concat([df_follow_edges, df_reply_resolved], ignore_index=True)

print(f"Total graph edges: {len(df_graph)}")
print(f"  follow edges: {len(df_follow_edges)}")
print(f"  reply edges:  {len(df_reply_resolved)}")

df_graph.to_csv(DATA_DIR + 'graph.csv', index=False)
print("Saved graph.csv")

Total graph edges: 133428
  follow edges: 127597
  reply edges:  5831
Saved graph.csv


In [28]:
# Keep only profiles whose author has at least one clean post
df_profiles = df_profiles[df_profiles['handle'].isin(df['handle'])].reset_index(drop=True)
print(f"Profiles after cleanup: {len(df_profiles)}")

df_profiles.to_csv(DATA_DIR + 'author_profiles.csv', index=False)
df.to_csv(DATA_DIR + 'posts_clean.csv', index=False)
print(f"Saved author_profiles.csv — {len(df_profiles)} profiles")
print(f"Saved posts_clean.csv — {len(df)} posts")

Profiles after cleanup: 2303
Saved author_profiles.csv — 2303 profiles
Saved posts_clean.csv — 3864 posts


In [29]:
# ── CONSISTENCY CHECKS ───────────────────────────────────────────

print("═" * 50)
print("POSTS CLEAN")
print("═" * 50)
print(f"Total posts:        {len(df)}")
print(f"Unique URIs:        {df['uri'].nunique()}  (should match total)")
print(f"Unique authors:     {df['handle'].nunique()}")
print(f"Null text:          {df['text_clean'].isnull().sum()}  (should be 0)")
print(f"Null handle:        {df['handle'].isnull().sum()}  (should be 0)")
print(f"Null created_at:    {df['created_at'].isnull().sum()}  (should be 0)")
print(f"Date range:         {df['created_at'].min()} → {df['created_at'].max()}")
print(f"User types:         {df['user_type'].value_counts().to_dict()}")

print("\n" + "═" * 50)
print("GRAPH")
print("═" * 50)
print(f"Total edges:        {len(df_graph)}")
print(f"Edge types:         {df_graph['type'].value_counts().to_dict()}")
print(f"Null source:        {df_graph['source'].isnull().sum()}  (should be 0)")
print(f"Null target:        {df_graph['target'].isnull().sum()}  (should be 0)")
print(f"Unique sources:     {df_graph['source'].nunique()}")
print(f"Unique targets:     {df_graph['target'].nunique()}")

print("\n" + "═" * 50)
print("PROFILES")
print("═" * 50)
print(f"Total profiles:     {len(df_profiles)}")
print(f"Null bios:          {df_profiles['bio'].isnull().sum()}")
print(f"All in posts:       {df_profiles['handle'].isin(df['handle']).all()}  (should be True)")

print("\n" + "═" * 50)
print("CROSS-FILE")
print("═" * 50)
graph_handles = set(df_graph['source']).union(set(df_graph['target']))
post_handles = set(df['handle'])
print(f"Graph handles in posts:  {len(graph_handles & post_handles)} / {len(graph_handles)}")
print(f"Reply sources in posts:  {df_reply_edges['source'].isin(post_handles).sum()} / {len(df_reply_edges)}")

══════════════════════════════════════════════════
POSTS CLEAN
══════════════════════════════════════════════════
Total posts:        3864
Unique URIs:        3864  (should match total)
Unique authors:     2303
Null text:          0  (should be 0)
Null handle:        0  (should be 0)
Null created_at:    0  (should be 0)
Date range:         2025-07-06 20:38:38+00:00 → 2026-07-06 08:47:37+00:00
User types:         {'general': 2843, 'professional': 610, 'unknown': 218, 'tech': 193}

══════════════════════════════════════════════════
GRAPH
══════════════════════════════════════════════════
Total edges:        133428
Edge types:         {'follows': 127597, 'reply': 5831}
Null source:        0  (should be 0)
Null target:        0  (should be 0)
Unique sources:     86982
Unique targets:     2088

══════════════════════════════════════════════════
PROFILES
══════════════════════════════════════════════════
Total profiles:     2303
Null bios:          118
All in posts:       True  (should be Tr

## Collection limitations

- **Ego-network sample.** Follow edges exist only around authors with ≥ 2 posts; accounts outside that set cannot have in-edges, so centrality comparisons are only valid within the post-authors subgraph.
- **Follower cap.** At most `FOLLOWERS_CAP` followers per author were collected (visible as a spike at the cap in the raw in-degree distribution).
- **Followers, not following.** We collect who follows the top authors, not who they follow.
- **Multilingual corpus.** Queries are English but thread replies include other languages; the relevance filter is keyword-based, not a language filter.
- **Handles as IDs.** Nodes are identified by handle; handles can change over time (DIDs are stable and are stored in the data as a fallback).
- **Ethics.** Only public posts and public follow lists were collected, at polite request rates, via the official API; credentials are kept in Colab secrets.